# Case-control LOBO model applied to the discordant sibling pairs

Author: Andre Macedo Date: 2026-07-06

Approach. Take the final case-control blood classifier (the LOBO "consensus" model, fit on
all 71 curated samples with the same pipeline: gene-level, DRIMSeq `~ Batch + Sex +
Diagnosis` feature selection, top-50, ridge) and apply it, unchanged, to the 9 in-house
discordant sibling pairs (18 samples), which were never part of the case-control cohort
(independent siblings_seq batches). Model + prediction script:
`analysis/blood/12-260706-LOBO_finalmodel_predict_siblings.R`.

Age control. Within each sibling pair, age is matched (mean |gap| 2.8 y) and both members are
young. If the classifier were merely an age detector, it could not separate proband from sibling
(both young -> both "ASD-like") and the within-pair ranking would be ~chance. If instead it ranks
the ASD proband above the unaffected sibling, it is detecting an ASD-specific signal beyond age.
Note: the absolute prob_ASD may be biased high for both members (the case-control model learned
ASD~young), so the within-pair ordering is the age-controlled read-out, not the raw level.

This is a harder test than the within-family LOPO (supp_9, Fig. 4F, 4G): there the model was
trained on sibling pairs; here the model has never seen a sibling pair and is transferred across
study designs.

Possible outcomes. A positive result (proband>sibling) would show the cohort model carries
ASD-specific, age-independent signal. A null/negative result would show the cohort model is
confound-driven and does not transfer to age-matched pairs, consistent with the sibling-pair
replication test (S5F, `07-260615-within_family_systematic_apa.ipynb`), where the cohort's per-gene directions do not
reproduce within families (concordance 0.42, below a random-gene null; effect-size Spearman -0.37).
Results are in the summary.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "apa-python.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/blood_samples/Classifications/2025/LOBO_CV")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import warnings

warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from scipy import stats
from itertools import product

RES = f"{DATA_ROOT}/3utr/Blood/2025_analyses/LOBO_finalmodel_predict_siblings"
ASD_C, CTRL_C, OK_C, BAD_C = "#d73027", "#4575b4", "#1a9850", "#d7191c"
preds = pd.read_csv(f"{RES}/sibling_preds.csv")

piv = preds.pivot_table(index="family", columns="truth", values="prob_ASD")
piv["margin"] = piv["ASD"] - piv["CTRL"]
piv["correct"] = piv["margin"] > 0
piv = piv.sort_values("ASD")
K, N = int(piv.correct.sum()), len(piv)

y = (preds.truth == "ASD").astype(int).values
fpr, tpr, _ = roc_curve(y, preds.prob_ASD.values)
A = auc(fpr, tpr)
# exact within-pair sign-flip permutation null for the pooled AUC (2^9 = 512)
pa, pc = piv["ASD"].values, piv["CTRL"].values
perm = []
for s in product([0, 1], repeat=N):
    s = np.array(s)
    asd = np.where(s == 0, pa, pc)
    ctr = np.where(s == 0, pc, pa)
    f2, t2, _ = roc_curve(np.r_[np.ones(N), np.zeros(N)], np.r_[asd, ctr])
    perm.append(auc(f2, t2))
p_perm = np.mean(np.array(perm) >= A)
binom_p = stats.binomtest(K, N, 0.5, alternative="greater").pvalue
print(f"within-pair proband>sibling: {K}/{N} (binomial p={binom_p:.4f})")
print(
    f"pooled AUC (proband vs sibling): {A:.3f} | exact within-pair permutation p={p_perm:.4f}"
)
print(
    f"absolute prob_ASD: proband mean {preds[preds.truth=='ASD'].prob_ASD.mean():.2f}, sibling mean {preds[preds.truth=='CTRL'].prob_ASD.mean():.2f}"
)

## Within-pair proband > sibling (case-control LOBO model)

Each line is a pair: the case-control model's predicted prob(ASD) for the ASD proband vs the unaffected sibling. Green = the model ranks the proband higher (correct); the vertical line marks the ASD~young absolute-probability bias (ignore the level, read the ordering).


In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.4))
for i, (fam, row) in enumerate(piv.iterrows()):
    ax.plot(
        [row["CTRL"], row["ASD"]],
        [i, i],
        color=OK_C if row["correct"] else BAD_C,
        lw=1.3,
        zorder=1,
    )
    ax.scatter(row["CTRL"], i, s=44, c=CTRL_C, edgecolors="k", linewidths=0.3, zorder=2)
    ax.scatter(row["ASD"], i, s=44, c=ASD_C, edgecolors="k", linewidths=0.3, zorder=2)
ax.axvline(0.5, ls=":", c="grey", lw=0.8)
ax.set_yticks(range(N))
ax.set_yticklabels(piv.index, fontsize=8)
ax.set_xlabel("predicted prob(ASD) - case-control LOBO model")
ax.set_ylabel("sibling pair (family)")
ax.set_title(f"Case-control model on sibling pairs: {K}/{N} proband>sibling")
ax.scatter([], [], s=44, c=ASD_C, edgecolors="k", linewidths=0.3, label="ASD proband")
ax.scatter(
    [], [], s=44, c=CTRL_C, edgecolors="k", linewidths=0.3, label="unaffected sibling"
)
ax.legend(fontsize=8, frameon=False, loc="lower right")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.savefig(f"{RES}/within_pair_dumbbell.png", dpi=150)
plt.show()
w = stats.wilcoxon(piv["ASD"], piv["CTRL"], alternative="greater")
print(
    f"within-pair {K}/{N} correct | one-sided Wilcoxon p={w.pvalue:.4f} | median within-pair margin={piv.margin.median():+.3f}"
)

## Pooled ROC (proband vs sibling)

The pooled ROC over the 18 samples (9 ASD-proband vs 9 CTRL-sibling), with an exact within-pair sign-flip permutation p (the appropriate null at n=9).


In [ ]:
rng = np.random.default_rng(0)
bs = []
for _ in range(2000):
    idx = rng.integers(0, len(preds), len(preds))
    if len(np.unique(y[idx])) < 2:
        continue
    f2, t2, _ = roc_curve(y[idx], preds.prob_ASD.values[idx])
    bs.append(auc(f2, t2))
lo, hi = np.percentile(bs, [2.5, 97.5])
fig, ax = plt.subplots(figsize=(4.2, 4.0))
ax.plot(fpr, tpr, color="#2c7fb8", lw=2)
ax.fill_between(fpr, tpr, 0, color="#2c7fb8", alpha=0.10)
ax.plot([0, 1], [0, 1], "--", c="grey", lw=0.8)
ax.text(
    0.33,
    0.12,
    f"AUC = {A:.2f}\n95% CI {lo:.2f}-{hi:.2f}\nperm p = {p_perm:.3f}",
    fontsize=9,
)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("Case-control model -> sibling pairs")
ax.set_aspect("equal")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.savefig(f"{RES}/pooled_roc_siblings.png", dpi=150)
plt.show()

## Cohort features with the sign re-learned within families (frozen features)

Freeze the 50 features selected by the cohort model (selected on the 71 case-control samples, independent of the siblings, so no leakage), but re-learn the ridge coefficients (sign) within families by leave-one-pair-out (train on 8 pairs, predict the held-out pair). If the cohort features carry age-independent within-family ASD information and only the cohort sign was confound-flipped, this recovers above-chance separation.


In [ ]:
import anndata as ad
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

PROPS_H5 = f"{DATA_ROOT}/3utr/Blood/all_till_20240724/utrcalling/UTR_count_table_genelevel_bw12_above10cts_apa_named_percentages.h5ad"
SHEET = f"{DATA_ROOT}/notebooks/blood_samples/Classifications/2025/LOPO_siblings/samplesheet_inhouse_9pairs.csv"
feat50 = pd.read_csv(f"{RES}/selected_features.csv")["feature_id"].tolist()
sheet = pd.read_csv(SHEET)
P = ad.read_h5ad(PROPS_H5)
keep = [f for f in feat50 if f in set(P.var_names)]
Xr = P[sheet.sample_id.tolist(), keep].X
Xr = np.nan_to_num(
    (Xr.toarray() if hasattr(Xr, "toarray") else np.asarray(Xr)).astype(float), nan=0.0
)
yb = (sheet.Diagnosis == "ASD").astype(int).values
fam = sheet.family.values
prob_rf = np.zeros(len(sheet))
for f in np.unique(fam):  # leave-one-pair-out; sign learned out-of-fold
    tr, te = fam != f, fam == f
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegressionCV(
            Cs=10,
            cv=5,
            penalty="l2",
            scoring="neg_log_loss",
            max_iter=5000,
            random_state=0,
        ),
    )
    clf.fit(Xr[tr], yb[tr])
    prob_rf[te] = clf.predict_proba(Xr[te])[:, 1]
rf = sheet[["family", "Diagnosis"]].copy()
rf["prob"] = prob_rf
pr2 = rf.pivot_table(index="family", columns="Diagnosis", values="prob")
Kr, Nr = int((pr2["ASD"] > pr2["CTRL"]).sum()), len(pr2)
Ar = auc(*roc_curve(yb, prob_rf)[:2])
pa2, pc2 = pr2["ASD"].values, pr2["CTRL"].values
perm2 = np.array(
    [
        auc(
            *roc_curve(
                np.r_[np.ones(Nr), np.zeros(Nr)],
                np.r_[
                    np.where(np.array(s) == 0, pa2, pc2),
                    np.where(np.array(s) == 0, pc2, pa2),
                ],
            )[:2]
        )
        for s in product([0, 1], repeat=Nr)
    ]
)
print(
    f"FROZEN cohort features + within-family-learned sign (LOPO), {len(keep)}/50 features:"
)
print(
    f"  within-pair proband>sibling: {Kr}/{Nr} | pooled AUC {Ar:.3f} | exact within-pair permutation p={np.mean(perm2 >= Ar):.4f}"
)
print(
    "  -> the SAME cohort features that failed on direct transfer separate the age-matched pairs once the sign is corrected"
)

In [ ]:
pr2

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.4))

families = pr2.index.tolist()
y_pos = np.arange(len(families))

for i, (fam, row) in enumerate(pr2.iterrows()):
    correct = row["ASD"] > row["CTRL"]
    ax.plot(
        [row["CTRL"], row["ASD"]],
        [i, i],
        color=OK_C if correct else BAD_C,
        lw=1.5,
        zorder=1,
    )
    ax.scatter(row["CTRL"], i, s=60, c=CTRL_C, edgecolors="k", linewidths=0.4, zorder=2)
    ax.scatter(row["ASD"], i, s=60, c=ASD_C, edgecolors="k", linewidths=0.4, zorder=2)

ax.axvline(0.5, ls=":", c="grey", lw=0.8, label="decision boundary")
ax.set_yticks(y_pos)
ax.set_yticklabels(families, fontsize=8)
ax.set_xlabel("predicted prob(ASD)\ncohort features + within-family sign (LOPO)")
ax.set_ylabel("sibling pair (family)")
ax.set_title(f"Frozen cohort features, refit sign: {Kr}/{Nr} proband > sibling")

ax.scatter([], [], s=60, c=ASD_C, edgecolors="k", linewidths=0.4, label="ASD proband")
ax.scatter(
    [], [], s=60, c=CTRL_C, edgecolors="k", linewidths=0.4, label="unaffected sibling"
)
ax.legend(fontsize=8, frameon=False, loc="lower left")

for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(f"{RES}/within_pair_dumbbell_refit.png", dpi=150)
plt.show()

## Three-way comparison: the sign differs, not the features


In [ ]:
s5i = pd.read_csv(
    f"{DATA_ROOT}/3utr/Blood/2025_analyses/LOPO_siblings_common/lopo_preds.csv"
)
A_s5i = auc(*roc_curve((s5i.truth == "ASD").astype(int), s5i.prob_ASD)[:2])
p5 = s5i.pivot_table(index="family", columns="truth", values="prob_ASD")
K_s5i = int((p5["ASD"] > p5["CTRL"]).sum())
conds = [
    "cohort features\n+ cohort sign\n(direct transfer)",
    "cohort features\n+ within-fam sign\n(refit LOPO)",
    "within-fam features\n+ within-fam sign\n(S5I LOPO)",
]
aucs, ks = [A, Ar, A_s5i], [K, Kr, K_s5i]
fig, ax = plt.subplots(figsize=(6.8, 4.2))
ax.bar(range(3), aucs, color=["#b2182b", "#f4a582", "#2166ac"], width=0.6)
ax.axhline(0.5, ls="--", c="grey", lw=0.8, label="chance")
for i, (a, k) in enumerate(zip(aucs, ks)):
    ax.text(i, a + 0.02, f"AUC {a:.2f}\n{k}/9", ha="center", fontsize=9)
ax.set_xticks(range(3))
ax.set_xticklabels(conds, fontsize=8)
ax.set_ylabel("within-pair pooled AUC")
ax.set_ylim(0, 1.0)
ax.legend(fontsize=8)
ax.set_title("What separates the sibling pairs: features vs sign")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.savefig(f"{RES}/features_vs_sign_comparison.png", dpi=150)
plt.show()
print(
    f"direct transfer {A:.3f} ({K}/9)  |  cohort features + refit sign {Ar:.3f} ({Kr}/9)  |  within-family {A_s5i:.3f} ({K_s5i}/9)"
)

## Summary: the cohort features carry ASD signal; only the sign is confound-flipped

- Direct transfer (cohort features + cohort sign): 3/9, AUC 0.32 (below chance); it mis-ranks
  age-matched pairs.
- Same 50 cohort features, sign re-learned within families (leave-one-pair-out, no leakage): 8/9,
  pooled AUC 0.69, exact within-pair permutation p ~ 0.006. The cohort features carry
  age-independent ASD information; only the cohort direction was wrong.
- Within-family features + within-family sign (supp_9 LOPO): 9/9, AUC 0.79.
- Interpretation. The case-control classifier's direction is set by the age confound
  (young-ASD-vs-old-CTRL reads as an apparent "ASD shortening" that is an age effect), which is
  opposite to the within-family ASD direction (global 3'UTR lengthening, notebook 07).
  Correcting the sign in age-matched families, the same features recover the signal. This links
  the confounded cohort signature (S5F non-replication) and the within-family effect: one
  ASD-relevant feature axis with a confound-flipped sign.
- No leakage: the 50 features come from the 71 case-control samples (no sibling data); the sign is
  learned out-of-fold across pairs.
- Caveat on the age-classifier control (S5J in an earlier figure version, not included here). Because age strongly shapes the direction of the
  cohort classifier (this experiment), the weaker performance of an age classifier is only a narrow
  specificity check ("not a coarse adult/child detector"), not evidence that the cohort signature is
  age-independent. The main evidence for an ASD effect beyond age is the age-matched within-family
  analyses (this refit, supp_9 LOPO, global within-family lengthening in notebook 07).
- Outputs in `.../2025_analyses/LOBO_finalmodel_predict_siblings/`.
